# Digital Addiction Project

## What's the problem?
People spend way too much time on their phones these days, and it's making them less productive.

## What are we doing?
I'm going to track how people use their phones, predict an addiction score, and then give some advice.

## Steps
1. Make some fake data.
2. Look at the graphs.
3. Predict the addiction score.
4. Group people into risk levels.
5. Give some advice.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

### 1. Making some fake data
Generating data for 500 random people.

In [2]:
np.random.seed(42)
n_samples = 500

# input data
daily_screen_time = np.random.uniform(1, 12, n_samples) 
social_media_usage = daily_screen_time * np.random.uniform(0.3, 0.8, n_samples)
productivity_usage = daily_screen_time * np.random.uniform(0.05, 0.2, n_samples)
sleep_time = 8 - (daily_screen_time * 0.2) + np.random.normal(0, 0.5, n_samples)
sleep_time = np.clip(sleep_time, 3, 10)

# formula for the score
# more phone = higher score, more sleep = lower score
raw_score = (daily_screen_time * 60) + (social_media_usage * 40) - (sleep_time * 30) - (productivity_usage * 20)
addiction_score = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min()) * 1000

df = pd.DataFrame({
    'daily_screen_time': daily_screen_time,
    'social_media_usage': social_media_usage,
    'productivity_usage': productivity_usage,
    'sleep_time': sleep_time,
    'addiction_score': addiction_score
})

print(df.head())

### 2. Looking at the data
Plotting some charts to see what's happening.

In [3]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlations")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(x='daily_screen_time', y='addiction_score', hue='social_media_usage', data=df, palette='viridis')
plt.title("Screen Time vs Addiction Score")
plt.show()

### 3. Predicting the addiction score
Training a model to guess the score.

In [4]:
X = df[['daily_screen_time', 'social_media_usage', 'productivity_usage', 'sleep_time']]
y = df['addiction_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

y_pred = reg_model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")

### 4. Grouping users by risk
Splitting users into 3 groups based on their usage.

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['daily_screen_time', 'addiction_score']])

kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# name the groups
cluster_means = df.groupby('cluster')['addiction_score'].mean().sort_values()
risk_mapping = {cluster_means.index[0]: 'Low', cluster_means.index[1]: 'Medium', cluster_means.index[2]: 'High'}
df['risk_level'] = df['cluster'].map(risk_mapping)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='daily_screen_time', y='addiction_score', hue='risk_level', data=df, palette='Set1')
plt.title("User Groups")
plt.show()

### 5. Giving advice based on score
Function to tell people how to fix their phone usage.

In [6]:
def give_advice(user_data):
    score = reg_model.predict([user_data])[0]
    
    if score < 300:
        level = "Low"
        suggestion = "You're doing great!"
    elif score < 700:
        level = "Medium"
        suggestion = f"Maybe use social media {user_data[1]*0.2:.1f} hours less."
    else:
        level = "High"
        suggestion = f"Too much phone! Try cutting down social media by {user_data[1]*0.5:.1f} hours."
    
    return score, level, suggestion

# trying it out
sample_user = [10.5, 8.0, 0.5, 4.5]
score, risk, advice = give_advice(sample_user)

print(f"Addiction Score: {score:.0f}/1000")
print(f"Risk Level: {risk}")
print(f"Advice: {advice}")